In [30]:
import pandas as pd
import numpy as np
from pathlib import Path

# Setup

In [31]:
# Data path
data_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data')

data_files_dict = {'index_1030' : ['index_1030 Gender Development Index.xlsx', '2025_metadata_Gender Development Index 2025.xlsx'],
                   'index_9' : ['index_9 EGDI.xlsx', '20251102_metadata_E-Government Development Index_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx', '20260710_metadata_Sustainable_Development_Index_2026.xlsx']}

index_name = 'index_9'
index_data_files = data_files_dict[index_name]

index_data_path = Path(data_path) / 'Historical' / index_data_files[0]
index_metadata_path = Path(data_path) / 'Metadata' / index_data_files[1]

# Read raw data
index_data_raw = pd.read_excel(index_data_path)
index_metadata_raw = pd.read_excel(index_metadata_path)
index_metadata_raw['Arab Countries'] = index_metadata_raw['Arab Countries'].str.strip()

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()
# - data correction
if index_name == 'index_1032':
    index_metadata_raw['Index_ID'] = '1032'
    index_metadata_raw = index_metadata_raw.merge(country_iso, how='left', left_on='Arab Countries', right_on='country')
    # make first row columns names
    index_data_raw.columns = index_data_raw.iloc[0]
    #index_data_raw = index_data_raw[1:]

In [32]:
arab_countries = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                  'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']
# Index data minor reformatting and filtering
index_data = index_data_raw.copy()
index_data = index_data.rename(columns={'Unnamed: 0':'KeyID','Unnamed: 1':'Series name','Unnamed: 2':'Year',
                                        'year':'Year'})
index_data = index_data.iloc[1:]

# Metadata
index_metadata = index_metadata_raw.copy()
index_metadata = index_metadata.rename(columns={'year of historical data': 'year historical',
                                                'year historical ':'year historical',
                                                'Year for each country, year of the simulator ':'Year for each country',
                                                'data simulator':'data'})

# Data consistency
index_metadata['year historical'] = (pd.to_numeric(index_metadata['year historical'], errors='coerce').astype('Int64'))
index_metadata['Year for each country'] = (pd.to_numeric(index_metadata['Year for each country'], errors='coerce').astype('Int64'))
index_metadata['year of the report'] = (pd.to_numeric(index_metadata['year of the report'], errors='coerce').astype('Int64'))
index_metadata['KPI ID'] = index_metadata['KPI ID'].astype(str)
index_metadata['KPI ID'] = index_metadata['KPI ID'].str.replace('.0','')

# Most recent historical data

In [33]:
# Composite key
newest_index_data = index_data.copy()

newest_index_data = newest_index_data.melt(id_vars=['KeyID','Year'], 
                                           var_name='Country', value_name='data_historical_automated')        # convert wide to long
newest_index_data = newest_index_data.dropna(subset=['data_historical_automated'])                            # drop na
newest_index_data = newest_index_data[newest_index_data['Country'].isin(arab_countries)] # filter for arab countries
# groupby and extract most recent year value
max_year_idx = newest_index_data.groupby(['KeyID', 'Country'])['Year'].idxmax()
newest_index_data = newest_index_data.loc[max_year_idx].reset_index(drop=True)

# Reformat values
newest_index_data['data_historical_automated'] = newest_index_data['data_historical_automated'].astype('Float64')

# Rename composite key (KeyID, Country) ==> (KPI_ID, ISO) to match metadata format
newest_index_data = newest_index_data.rename(columns={'KeyID':'KPI ID', 'Country':'ISO', 'Year':'year_historical_automated'})
newest_index_data['KPI ID'] = newest_index_data['KPI ID'].astype(int).astype(str)

In [34]:
# Composite key
data_long_format = index_data.copy()

data_long_format = data_long_format.melt(id_vars=['KeyID','Year'], 
                                           var_name='Country', value_name='data_historical_automated')        # convert wide to long
data_long_format = data_long_format.dropna(subset=['data_historical_automated'])                            # drop na
data_long_format = data_long_format[data_long_format['Country'].isin(arab_countries)]

In [35]:
index_metadata_kpis = index_metadata['KPI ID'].unique().tolist()
historical_index_kpis = newest_index_data['KPI ID'].unique().tolist()
result = list(set(index_metadata_kpis) - set(historical_index_kpis))
result

[]

In [36]:
newest_index_data

,KPI ID,year_historical_automated,ISO,data_historical_automated
0,271,2023,ARE,199.424
1,271,2023,BHR,153.9
2,271,2023,COM,109.856
3,271,2023,DJI,48.4505
4,271,2023,DZA,111.606
...,...,...,...,...
309,1942,2024,SDN,8.22
310,1942,2024,SOM,59.02
311,1942,2024,SYR,3.91
312,1942,2024,TUN,4.17


# Recreate metadata

In [37]:
# Use current file as template
metadata_automated = index_metadata[['Index_ID','KPI name in the report ','Arab Countries','ISO','KPI ID','Year for each country', 'year of the report', 
                                     'data', 'year historical', 'data historical', 'Data availability and treatment',
                                    'Year difference', 'last year available used to compute the index', 'Value difference',
                                    'Accuracy problem']].copy()

# join in automated data pull
metadata_automated = metadata_automated.merge(newest_index_data, how='left', on=['KPI ID', 'ISO'])
# Recreate calculated fields
# G 'Year for each country'
# I 'year of the report'
#metadata_check = index_metadata_check.merge(metadata_recreate[['KPI ID', 'ISO', 'Year', 'data']], how='left', on=['KPI ID','ISO'])

In [38]:
# Data Availability and treatment
index_metadata.columns

Index(['Index_ID', 'KPI name in the report ', 'Arab Countries', 'ISO',
       'Nom de l'index', 'Pillar', 'Justificatin for the pillar', 'KPI ID',
       'Year for each country', 'data historical', 'year of the report',
       'data', 'Definition in the report ', 'Unit in the report', 'Source',
       'URL', 'KPI name in the source', 'Definition in the source',
       'Unit in the source', 'year historical', 'data historical.1', 'SDG',
       'Data availability and treatment', 'Year difference',
       'last year available used to compute the index', 'Value difference',
       'Accuracy problem'],
      dtype='str')

### Calculated field alternative approach

In [39]:
G_col = 'Year for each country'
I_col = 'year of the report'
### Automated fields (from historical KPI file) ###
H_col = 'data historical' #'data_historical_automated'
R_col = 'year historical' #'year_historical_automated'
### Computed fields ###
U_col = 'Data availability and treatment_computed'
V_col = 'Year difference_computed'
W_col = 'last year available used to compute the index_computed'
X_col = 'Value difference_computed'
Y_col = 'Accuracy problem_computed'

In [40]:
# U - Data availability and treatment
metadata_automated['G_col_available'] = np.where(metadata_automated[G_col].isna(), 0, 1)
metadata_automated['I_col_available'] = np.where(metadata_automated[I_col].isna(), 0, 1)

conditions_U = [
    (metadata_automated['G_col_available']==0) & (metadata_automated['I_col_available']==0),   # Both I and G are empty
    (metadata_automated['G_col_available']==0) & (metadata_automated['I_col_available']==1),  # G empty, I available
    (metadata_automated['G_col_available']==1) & (metadata_automated['I_col_available']==1)  # Both I and G are available
]
condition_outputs_U = ['No data and not included',
                       'No Data available, was imputed',
                       'Data available']
metadata_automated[U_col] = np.select(conditions_U, condition_outputs_U, default='Data available but not included')

In [41]:
# V - Year Difference
G = metadata_automated[G_col]
R = metadata_automated[R_col]
I = metadata_automated[I_col]

G_empty = G.isna() | (G.astype(str).str.strip() == '')
I_empty = I.isna() | (I.astype(str).str.strip() == '')

diff = pd.to_numeric(R, errors='coerce') - pd.to_numeric(G, errors='coerce')
diff_str = diff.map(lambda x: '' if pd.isna(x) else str(int(x)))
metadata_automated[V_col] = np.where(G_empty | I_empty, '', diff_str)

In [42]:
# W - Last Year available used to compute index
conditions_W = [
    metadata_automated[V_col]=='',   # V is empty
    metadata_automated[V_col]=='0'   # V is 0, up to date
]
condition_outputs_U = ['',
                       'YES']
metadata_automated[W_col] = np.select(conditions_W, condition_outputs_U, default='NO')

In [43]:
# X - Value Difference
metadata_automated[X_col] = (pd.to_numeric(metadata_automated[H_col],errors='coerce')-pd.to_numeric(metadata_automated['data'], errors='coerce'))

In [44]:
# Y - Accuracy Problem
status = metadata_automated[U_col]
vdiff  = metadata_automated[X_col]   # float64

conditions_Y = [
    status == 'Data available but not included',
    status == 'No data and not included',
    status == 'No Data available, was imputed',
    (status == 'Data available') & ((vdiff == 0) | vdiff.isna()),   # blank/0 -> NO, matches Excel
    (status == 'Data available') & (vdiff != 0) & vdiff.notna(),    # real nonzero -> YES
]
condition_outputs_Y = ['', '', 'IMP', 'NO', 'YES']

metadata_automated[Y_col] = np.select(
    conditions_Y, condition_outputs_Y, default='YES'
)

In [45]:
# Discrepancies
U_col_discrepancy = metadata_automated[['ISO', 'KPI ID',G_col,I_col,'G_col_available','I_col_available','Data availability and treatment',U_col]].copy()
U_col_discrepancy = U_col_discrepancy[U_col_discrepancy['Data availability and treatment'] != U_col_discrepancy[U_col]].copy()
print(f'U_col discrepancies - {U_col_discrepancy.shape}')

V_col_discrepancy = metadata_automated[['ISO', 'KPI ID',G_col,I_col,'Year difference',V_col]].copy()
V_col_discrepancy['Year difference'] = V_col_discrepancy['Year difference'].astype(int, errors='ignore').astype(str).fillna('').str.replace('.0','')
V_col_discrepancy = V_col_discrepancy[V_col_discrepancy['Year difference'].astype(str) != V_col_discrepancy[V_col].astype(str)].copy()
print(f'V_col discrepancies - {V_col_discrepancy.shape}')

W_col_discrepancy = metadata_automated[['ISO','KPI ID','Year difference',V_col,
                                        'last year available used to compute the index',W_col]].copy()
W_col_discrepancy['last year available used to compute the index'] = W_col_discrepancy['last year available used to compute the index'].astype(str).fillna('')
W_col_discrepancy = W_col_discrepancy[W_col_discrepancy['last year available used to compute the index'].astype(str) != W_col_discrepancy[W_col].astype(str)]
print(f'W_col discrepancies - {W_col_discrepancy.shape}')

X_col_discrepancy = metadata_automated[['ISO','KPI ID',H_col,'data', 'Value difference',X_col]].copy()
X_col_discrepancy['Value difference'] = X_col_discrepancy['Value difference'].astype(int,errors='ignore').fillna('')
X_col_discrepancy[X_col] = X_col_discrepancy[X_col].astype(int,errors='ignore').fillna('')
X_col_discrepancy = X_col_discrepancy[X_col_discrepancy['Value difference'].astype(str) != X_col_discrepancy[X_col].astype(str)].copy()
print(f'X_col discrepancies - {X_col_discrepancy.shape}')

Y_col_discrepancy = metadata_automated[['ISO','KPI ID','Data availability and treatment','Value difference','Accuracy problem',Y_col,U_col,X_col]].copy()
Y_col_discrepancy = Y_col_discrepancy[Y_col_discrepancy['Accuracy problem'].astype(str) != Y_col_discrepancy[Y_col].astype(str)]
print(f'Y_col discrepancies - {Y_col_discrepancy.shape}')


U_col discrepancies - (21, 8)
V_col discrepancies - (0, 6)
W_col discrepancies - (0, 6)
X_col discrepancies - (0, 6)
Y_col discrepancies - (36, 8)


In [46]:
U_col_discrepancy

,ISO,KPI ID,Year for each country,year of the report,G_col_available,I_col_available,Data availability and treatment,Data availability and treatment_computed
294,DZA,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"
295,BHR,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"
296,COM,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"
297,DJI,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"
298,EGY,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"
299,IRQ,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"
300,JOR,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"
301,KWT,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"
302,LBN,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"
303,LBY,271,<NA>,2024,0,1,NaN,"No Data available, was imputed"


## Summary stats

binary 1/0
Perfect (up to date data), Data from prior month, Imputed data/Value difference

In [47]:
data_hist_col = 'data historical'
year_hist_col = 'year historical'

metadata_automated['value_difference'] = abs(metadata_automated['data']-metadata_automated[data_hist_col])
metadata_automated['value_difference_percentage'] = np.where(
    (metadata_automated['value_difference'] == 0) & (metadata_automated['data'] == 0),
    0,
    metadata_automated['value_difference'] / metadata_automated['data']
)

metadata_automated['year_difference'] = metadata_automated[year_hist_col]-metadata_automated['Year for each country']

metadata_automated['year_equal'] = ((metadata_automated['year_difference']==0).fillna(False))
metadata_automated['value_equal'] = (
    (metadata_automated['value_difference_percentage'] < 0.01).fillna(False)
)
metadata_automated['Record_Quality_Good'] = np.where(metadata_automated['year_equal'] & metadata_automated['value_equal'],
                                                    1,
                                                    0)
metadata_automated['Record_Quality_Old'] = np.where(metadata_automated['year_difference'].fillna(0) != 0,
                                                    1,
                                                    0)
metadata_automated['Record_Quality_Inaccurate'] = np.where(metadata_automated['year_equal'] & ~metadata_automated['value_equal'],
                                                    1,
                                                    0)
metadata_automated['Record_Quality_Imputed'] = np.where(metadata_automated['Year for each country'].isna(),
                                                        1,
                                                        0)
metadata_automated['Record_Quality_Total'] = metadata_automated['Record_Quality_Good']+metadata_automated['Record_Quality_Old']+metadata_automated['Record_Quality_Inaccurate']+metadata_automated['Record_Quality_Imputed']


In [48]:
total_quality_good = sum(metadata_automated['Record_Quality_Good'])
total_quality_old = sum(metadata_automated['Record_Quality_Old'])
total_quality_inaccurate = sum(metadata_automated['Record_Quality_Inaccurate'])
total_quality_imputed = sum(metadata_automated['Record_Quality_Imputed'])

total_records = total_quality_good+total_quality_old+total_quality_inaccurate+total_quality_imputed


if metadata_automated.shape[0] == total_records:
    print(f"All {metadata_automated.shape[0]} records accounted for.")
else:
    print(f"Record count mismatch {total_records} of {metadata_automated.shape[0]}")

summary_stats = pd.DataFrame({
    'Category': ['Good', 'Old', 'Inaccurate', 'Imputed'],
    'Count': [total_quality_good, total_quality_old, total_quality_inaccurate, total_quality_imputed],
})
summary_stats['Percentage'] = (summary_stats['Count'] / total_records * 100).round(2)
# Add a totals row
summary_stats.loc['Total'] = ['Total', total_records, 100.0]

All 315 records accounted for.


In [49]:
summary_stats_kpi = metadata_automated.groupby(['KPI ID'])[
    ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate', 'Record_Quality_Imputed']
].sum().reset_index()

summary_stats_kpi['Total'] = (
    summary_stats_kpi['Record_Quality_Good']
    + summary_stats_kpi['Record_Quality_Old']
    + summary_stats_kpi['Record_Quality_Inaccurate']
    + summary_stats_kpi['Record_Quality_Imputed']
)

# Add percentage columns
for col in ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate', 'Record_Quality_Imputed']:
    summary_stats_kpi[col + '_pct'] = (summary_stats_kpi[col] / summary_stats_kpi['Total']).round(2)

In [50]:
summary_stats_country = metadata_automated.groupby(['Arab Countries'])[
    ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate', 'Record_Quality_Imputed']
].sum().reset_index()

summary_stats_country['Total'] = (
    summary_stats_country['Record_Quality_Good']
    + summary_stats_country['Record_Quality_Old']
    + summary_stats_country['Record_Quality_Inaccurate']
    + summary_stats_country['Record_Quality_Imputed']
)

# Add percentage columns
for col in ['Record_Quality_Good', 'Record_Quality_Old', 'Record_Quality_Inaccurate', 'Record_Quality_Imputed']:
    summary_stats_country[col + '_pct'] = (summary_stats_country[col] / summary_stats_country['Total']).round(2)

In [51]:
summary_stats_country

,Arab Countries,Record_Quality_Good,Record_Quality_Old,Record_Quality_Inaccurate,Record_Quality_Imputed,Total,Record_Quality_Good_pct,Record_Quality_Old_pct,Record_Quality_Inaccurate_pct,Record_Quality_Imputed_pct
0,Algeria,11,3,0,1,15,0.73,0.20,0.00,0.07
1,Bahrain,9,4,0,2,15,0.60,0.27,0.00,0.13
2,Comoros,11,3,0,1,15,0.73,0.20,0.00,0.07
3,Djibouti,10,3,0,2,15,0.67,0.20,0.00,0.13
4,Egypt,11,3,0,1,15,0.73,0.20,0.00,0.07
5,Iraq,11,3,0,1,15,0.73,0.20,0.00,0.07
6,Jordan,10,3,0,2,15,0.67,0.20,0.00,0.13
7,Kuwait,11,3,0,1,15,0.73,0.20,0.00,0.07
8,Lebanon,10,2,1,2,15,0.67,0.13,0.07,0.13
9,Libya,10,2,1,2,15,0.67,0.13,0.07,0.13


In [53]:

from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
def write_dataframes_to_excel(
    dataframes: dict,
    output_path: str = "output.xlsx",
    freeze_header: bool = True,
    autofit_columns: bool = True,
    max_col_width: int = 50,
    header_bg_color: str = None,#"4472C4",   # Excel-blue; set None for no fill
    header_font_color: str = "FFFFFF",
    header_bold: bool = True,
    add_borders: bool = False,
    add_autofilter: bool = True,
    index: bool = False,
):
    """
    Write a dict of DataFrames to one Excel file, one sheet per DataFrame.
 
    Parameters
    ----------
    dataframes : dict
        Mapping of {sheet_name: dataframe}. Sheet names are what you control
        naming for — just use whatever keys you want.
        Example: {"Sales": df_sales, "Inventory": df_inv}
    output_path : str
        Path to save the .xlsx file.
    freeze_header : bool
        If True, freezes the header row so it stays visible when scrolling.
    autofit_columns : bool
        If True, resizes each column to fit its longest value.
    max_col_width : int
        Cap on auto-fit column width (prevents huge columns from long text).
    header_bg_color : str or None
        Hex color (no '#') for header cell background fill.
    header_font_color : str
        Hex color (no '#') for header font color.
    header_bold : bool
        Whether header text is bold.
    add_borders : bool
        If True, adds thin borders around all cells.
    add_autofilter : bool
        If True, adds filter dropdowns to the header row.
    index : bool
        Whether to write the DataFrame index as a column.
    """
 
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dataframes.items():
            # Excel sheet names max 31 chars, no : \ / ? * [ ]
            safe_name = _sanitize_sheet_name(sheet_name)
            df.to_excel(writer, sheet_name=safe_name, index=index)
 
            worksheet = writer.sheets[safe_name]
            _format_worksheet(
                worksheet,
                df,
                freeze_header=freeze_header,
                autofit_columns=autofit_columns,
                max_col_width=max_col_width,
                header_bg_color=header_bg_color,
                header_font_color=header_font_color,
                header_bold=header_bold,
                add_borders=add_borders,
                add_autofilter=add_autofilter,
                index=index,
            )
 
    print(f"Saved: {output_path}")
 
 
def _sanitize_sheet_name(name: str) -> str:
    invalid_chars = [":", "\\", "/", "?", "*", "[", "]"]
    for ch in invalid_chars:
        name = name.replace(ch, "")
    return name[:31]
 
 
def _format_worksheet(
    ws,
    df,
    freeze_header,
    autofit_columns,
    max_col_width,
    header_bg_color,
    header_font_color,
    header_bold,
    add_borders,
    add_autofilter,
    index,
):
    n_cols = df.shape[1] + (1 if index else 0)
    n_rows = df.shape[0] + 1  # +1 for header row
 
    # --- Header styling ---
    header_font = Font(bold=header_bold, color=header_font_color if header_bg_color else "000000")
    header_fill = PatternFill(
        start_color=header_bg_color, end_color=header_bg_color, fill_type="solid"
    ) if header_bg_color else None
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
 
    for col_idx in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        if header_fill:
            cell.fill = header_fill
        cell.alignment = header_align
 
    # Slightly taller header row to fit wrapped text
    ws.row_dimensions[1].height = 22
 
    # --- Optional thin borders on all used cells ---
    if add_borders:
        thin = Side(style="thin", color="B7B7B7")
        border = Border(left=thin, right=thin, top=thin, bottom=thin)
        for row in ws.iter_rows(min_row=1, max_row=n_rows, min_col=1, max_col=n_cols):
            for cell in row:
                cell.border = border
 
    # --- Freeze header row (and index column if present) ---
    if freeze_header:
        ws.freeze_panes = "B2" if index else "A2"
 
    # --- Autofilter on header row ---
    if add_autofilter:
        ws.auto_filter.ref = ws.dimensions
 
    # --- Autofit column widths based on longest value ---
    if autofit_columns:
        for col_idx in range(1, n_cols + 1):
            col_letter = get_column_letter(col_idx)
            max_len = 0
            for cell in ws[col_letter]:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[col_letter].width = min(max_len + 3, max_col_width)

In [54]:

output_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data') / 'metadata_test.xlsx'

sheets = {
    'Metadata_automated' : metadata_automated,
    'Summary Stats' : summary_stats,
    'Summary Stats by KPI' : summary_stats_kpi,
    'Summary Stats by Country' : summary_stats_country,
    'Raw Data Long format' : data_long_format
}

write_dataframes_to_excel(
        sheets,
        output_path=output_path,
        freeze_header=True,
        autofit_columns=True,
        #header_bg_color="4472C4",
        #header_font_color="FFFFFF",
        header_bold=True,
        add_borders=True,
        add_autofilter=True,
        index=False,
    )

Saved: C:\Users\User\OneDrive\Desktop\UN\Data\metadata_test.xlsx
